In [ ]:
# ── STEP 1: Force-reinstall PyTorch with CUDA 12.1 (matches Kaggle's current driver) ──
!pip install -q --upgrade torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu121

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    x = torch.zeros(3, 3).cuda()
    x.fill_(1.0)
    print("✅ CUDA kernel test passed!")
else:
    print("❌ No GPU — make sure Accelerator is set to GPU in notebook settings")


In [ ]:
# ── STEP 2: System deps ──
!apt-get install -y poppler-utils 2>/dev/null | tail -3

# ── STEP 3: Core packages ──
!pip install -q \
    "fastapi==0.115.6" \
    "uvicorn[standard]==0.34.0" \
    "python-multipart==0.0.20" \
    "pyngrok==7.2.2" \
    "nest-asyncio==1.6.0" \
    "Pillow==11.1.0" \
    "pdf2image==1.17.0" \
    "openpyxl==3.1.5" \
    "pandas==2.2.3" \
    "python-docx==1.1.2" \
    "matplotlib"

# ── ChromaDB as the vector database ──
# We pass embedding_function=None so chromadb never downloads
# sentence-transformers or onnxruntime — ColPali supplies all vectors.
!pip install -q "chromadb>=0.5.0"

# ── byaldi last — reuses the torch already installed ──
!pip install -q "byaldi==0.0.7"

print("✅ All dependencies installed!")


In [ ]:
# # ── Pin dependencies to avoid architecture conflicts ──
# !pip install -q \
#     "numpy<2.0.0" \
#     "transformers==4.46.3" \
#     "peft==0.12.0" \
#     "colpali-engine==0.3.1"

# ── Pin dependencies to avoid architecture conflicts ──
!pip install -q \
    "numpy<2.0.0" \
    "transformers>=4.49.0" \
    "peft>=0.12.0" \
    "accelerate>=0.27.0" \
    "colpali-engine==0.3.1"

In [1]:
import os, json, uuid, time, shutil, base64, asyncio, logging
from pathlib import Path
from typing import Optional, List, Dict, Any
from io import BytesIO
from datetime import datetime

import nest_asyncio
nest_asyncio.apply()

import torch
import pandas as pd
from PIL import Image
from pdf2image import convert_from_path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from fastapi import FastAPI, UploadFile, File, Form, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, StreamingResponse
import uvicorn
from pyngrok import ngrok

# ── Logging ──
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger("BuildMarshalAI")

# ── Paths ──
BASE_DIR   = Path("/kaggle/working/buildmarshal")
DOCS_DIR   = BASE_DIR / "documents"
PAGES_DIR  = BASE_DIR / "pages"
INDEX_DIR  = BASE_DIR / "index"
CHROMA_DIR = BASE_DIR / "chroma_db"    # ChromaDB persistence directory
META_FILE  = BASE_DIR / "metadata.json"

for d in [BASE_DIR, DOCS_DIR, PAGES_DIR, INDEX_DIR, CHROMA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── API Keys via Kaggle Secrets ──
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    NGROK_AUTH_TOKEN = secrets.get_secret("NGROK_AUTH_TOKEN")
    logger.info("✅ Loaded secrets from Kaggle Secrets")
except Exception as e:
    logger.warning(f"Kaggle Secrets unavailable ({e}). Falling back to env vars.")
    NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN", "YOUR_NGROK_TOKEN_HERE")

assert NGROK_AUTH_TOKEN != "YOUR_NGROK_TOKEN_HERE", "❌ Set your NGROK_AUTH_TOKEN!"

# ── Metadata helpers ──
def load_metadata() -> Dict:
    if META_FILE.exists():
        return json.loads(META_FILE.read_text())
    return {"documents": {}}

def save_metadata(meta: Dict):
    META_FILE.write_text(json.dumps(meta, indent=2, default=str))

print("✅ Config ready | GPU:", torch.cuda.is_available())


2026-04-04 21:18:00,174 | INFO | ✅ Loaded secrets from Kaggle Secrets


✅ Config ready | GPU: True


In [3]:
# =======================================================================
# CELL 5 — ColPali  +  ChromaDB vector store
# =======================================================================

import gc, torch, numpy as np
import chromadb
from typing import Optional, List, Dict
from PIL import Image

# ── PEFT / transformers KeyError: 'llava' patch ──
# PaliGemma uses a llava-style architecture. When transformers' PEFT integration
# tries to look it up in _MOE_TARGET_MODULE_MAPPING it fails with KeyError: 'llava'.
# These two patches make that lookup a safe no-op.
import transformers.integrations.peft as _peft_integ

_orig_moe = _peft_integ._convert_peft_config_moe
def _safe_moe(p, m):
    try: return _orig_moe(p, m)
    except KeyError: return p
_peft_integ._convert_peft_config_moe = _safe_moe

_orig_cvt = _peft_integ.convert_peft_config_for_transformers
def _safe_cvt(p, model=None, conversions=None):
    try: return _orig_cvt(p, model=model, conversions=conversions)
    except KeyError: return p
_peft_integ.convert_peft_config_for_transformers = _safe_cvt

print("✅ PEFT patch applied")

# ── Free VRAM before loading ──
for _v in ['RAG_MODEL', 'COLPALI_MODEL', 'COLPALI_PROCESSOR']:
    if _v in dir() and eval(_v) is not None:
        exec(f"del {_v}")
gc.collect()
torch.cuda.empty_cache()
print(f"Free VRAM before load: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

# ── Load ColPali using colpali_engine's own classes ──
from colpali_engine.models import ColPali, ColPaliProcessor

ADAPTER_ID = "vidore/colpali-v1.2"

logger.info("Loading ColPali v1.2 (adapter-merged checkpoint)...")
COLPALI_MODEL = ColPali.from_pretrained(
    ADAPTER_ID,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    low_cpu_mem_usage=True,
).eval()

COLPALI_PROCESSOR = ColPaliProcessor.from_pretrained(ADAPTER_ID)

print(f"✅ ColPali ready. Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

RAG_MODEL = None

# =======================================================================
# ChromaDB — persistent vector store with per-page metadata
# =======================================================================

_COLLECTION_NAME = "document_pages"

CHROMA_CLIENT = chromadb.PersistentClient(path=str(CHROMA_DIR))

try:
    CHROMA_COLLECTION = CHROMA_CLIENT.get_collection(
        name=_COLLECTION_NAME,
        embedding_function=None,
    )
    logger.info(
        f"✅ Loaded existing ChromaDB collection "
        f"({CHROMA_COLLECTION.count()} pages indexed)"
    )
except Exception:
    CHROMA_COLLECTION = CHROMA_CLIENT.create_collection(
        name=_COLLECTION_NAME,
        embedding_function=None,
        metadata={"hnsw:space": "cosine"},
    )
    logger.info("✅ Created new ChromaDB collection")


@torch.no_grad()
def embed_image(image_path: str) -> Optional[np.ndarray]:
    try:
        img = Image.open(image_path).convert("RGB")

        # process_images() builds the correct token structure for page screenshots
        batch = COLPALI_PROCESSOR.process_images([img]).to(COLPALI_MODEL.device)

        # ColPali.forward() returns a raw Tensor (batch, seq_len, dim) — NOT a ModelOutput
        out = COLPALI_MODEL(**batch)           # shape: (1, seq_len, embedding_dim)
        vec = out.mean(dim=1).squeeze(0)       # mean-pool → (embedding_dim,)

        return vec.float().cpu().numpy()
    except Exception as e:
        logger.error(f"embed_image failed for {image_path}: {e}")
        return None


@torch.no_grad()
def embed_query(query: str) -> Optional[np.ndarray]:
    try:
        # process_queries() builds the correct token structure for text queries
        batch = COLPALI_PROCESSOR.process_queries([query]).to(COLPALI_MODEL.device)

        # Same — ColPali returns a raw Tensor
        out = COLPALI_MODEL(**batch)           # shape: (1, seq_len, embedding_dim)
        vec = out.mean(dim=1).squeeze(0)       # mean-pool → (embedding_dim,)

        return vec.float().cpu().numpy()
    except Exception as e:
        logger.error(f"embed_query failed: {e}")
        return None


def retrieve_context(query: str, top_k: int = 5) -> List[Dict]:
    if CHROMA_COLLECTION.count() == 0:
        logger.warning("ChromaDB collection empty — falling back to all pages")
        return get_all_pages(top_k)

    q_vec = embed_query(query)
    if q_vec is None:
        logger.error("Query embedding failed — falling back to all pages")
        return get_all_pages(top_k)

    n = min(top_k, CHROMA_COLLECTION.count())
    results = CHROMA_COLLECTION.query(
        query_embeddings=[q_vec.tolist()],
        n_results=n,
        include=["metadatas", "distances"],
    )

    pages = []
    for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
        similarity = round(1.0 - dist, 4)
        pages.append({
            "doc_name":     meta["doc_name"],
            "doc_id":       meta["doc_id"],
            "page":         int(meta["page_num"]),
            "image_path":   meta.get("image_path", ""),
            "text_content": meta.get("text_content", ""),
            "score":        similarity,
        })

    if pages:
        logger.info(f"Retrieved {len(pages)} pages (top similarity: {pages[0]['score']})")
    return pages


def get_all_pages(limit: int = 5) -> List[Dict]:
    meta, pages = load_metadata(), []
    for doc_id, doc in meta["documents"].items():
        for page in doc["pages"]:
            pages.append({
                "doc_name":     doc["name"],
                "doc_id":       doc_id,
                "page":         page["page_num"],
                "image_path":   page.get("image_path", ""),
                "text_content": page.get("text_content", ""),
                "score":        0.0,
            })
            if len(pages) >= limit:
                return pages
    return pages


print("✅ ColPali + ChromaDB vector store ready")
print(f"   Collection '{_COLLECTION_NAME}' — {CHROMA_COLLECTION.count()} pages indexed")

2026-04-04 21:19:36,720 | INFO | Loading ColPali v1.2 (adapter-merged checkpoint)...
2026-04-04 21:19:36,851 | INFO | HTTP Request: HEAD https://huggingface.co/vidore/colpali-v1.2/resolve/main/config.json "HTTP/1.1 404 Not Found"


✅ PEFT patch applied
Free VRAM before load: 78.48 GB


2026-04-04 21:19:36,957 | INFO | HTTP Request: HEAD https://huggingface.co/vidore/colpali-v1.2/resolve/main/adapter_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-04 21:19:36,965 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/vidore/colpali-v1.2/6b89bc63c16809af4d111bfe412e2ac6bc3c9451/adapter_config.json "HTTP/1.1 200 OK"
2026-04-04 21:19:37,067 | INFO | HTTP Request: HEAD https://huggingface.co/vidore/colpaligemma-3b-pt-448-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-04 21:19:37,075 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/vidore/colpaligemma-3b-pt-448-base/30ab955d073de4a91dc5a288e8c97226647e3e5a/config.json "HTTP/1.1 200 OK"
2026-04-04 21:19:37,187 | INFO | HTTP Request: HEAD https://huggingface.co/vidore/colpaligemma-3b-pt-448-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-04-04 21:19:37,290 | INFO | HTTP Request: HEAD https://huggingface.co/vidore/colpaligemma-3b-

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/605 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

2026-04-04 21:19:42,550 | INFO | HTTP Request: HEAD https://huggingface.co/vidore/colpali-v1.2/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
2026-04-04 21:19:42,651 | INFO | HTTP Request: HEAD https://huggingface.co/vidore/colpali-v1.2/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
2026-04-04 21:19:42,660 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/vidore/colpali-v1.2/6b89bc63c16809af4d111bfe412e2ac6bc3c9451/special_tokens_map.json "HTTP/1.1 200 OK"
2026-04-04 21:19:42,670 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/vidore/colpali-v1.2/6b89bc63c16809af4d111bfe412e2ac6bc3c9451/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

2026-04-04 21:19:43,586 | INFO | HTTP Request: GET https://huggingface.co/api/models/vidore/colpali-v1.2 "HTTP/1.1 200 OK"
2026-04-04 21:19:43,657 | INFO | ✅ Loaded existing ChromaDB collection (0 pages indexed)


✅ ColPali ready. Free VRAM: 66.58 GB
✅ ColPali + ChromaDB vector store ready
   Collection 'document_pages' — 0 pages indexed


In [4]:
# =======================================================================
# CELL 6 — Document Ingestion  ->  ColPali embedding  ->  ChromaDB
# =======================================================================
#
# Flow for each file:
#   convert to page screenshots
#   for each page:
#       embed_image(screenshot)  -> float32 numpy vector
#       CHROMA_COLLECTION.add(id, embedding, metadata, document)
#   persist metadata.json (for /api/documents listing and deletion)
# =======================================================================

def process_pdf(file_path: Path, doc_id: str) -> List[Dict]:
    logger.info(f"Processing PDF: {file_path.name}")
    pages = convert_from_path(str(file_path), dpi=150, fmt='png')
    page_infos = []
    for i, page_img in enumerate(pages):
        page_path = PAGES_DIR / f"{doc_id}_page_{i+1}.png"
        page_img.thumbnail((1024, 1024))
        page_img.save(str(page_path), "PNG")
        page_infos.append({
            "page_num":    i + 1,
            "image_path":  str(page_path),
            "width":       page_img.width,
            "height":      page_img.height,
            "text_content": "",
        })
    logger.info(f"PDF: {len(pages)} pages extracted")
    return page_infos


def _df_to_image(df, title, out_path):
    fig, ax = plt.subplots(figsize=(min(20, len(df.columns)*1.5+1),
                                    min(20, len(df)*0.4+1)))
    ax.axis('off')
    data = df.head(50).fillna('').astype(str)
    tbl  = ax.table(cellText=data.values, colLabels=data.columns,
                    cellLoc='left', loc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.3)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor('#4472C4')
            cell.set_text_props(color='white', fontweight='bold')
        else:
            cell.set_facecolor('#f0f0f0' if r % 2 == 0 else 'white')
    plt.title(title, fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(out_path), dpi=120, bbox_inches='tight', facecolor='white')
    plt.close(fig)


def process_excel(file_path: Path, doc_id: str) -> List[Dict]:
    page_infos = []
    dfs = ({"Sheet1": pd.read_csv(str(file_path))}
           if file_path.suffix == '.csv'
           else pd.read_excel(str(file_path), sheet_name=None))
    for sheet_name, df in dfs.items():
        page_path = PAGES_DIR / f"{doc_id}_sheet_{sheet_name}.png"
        try:
            _df_to_image(df, f"{file_path.stem} — {sheet_name}", page_path)
            page_infos.append({
                "page_num":    len(page_infos) + 1,
                "image_path":  str(page_path),
                "sheet_name":  sheet_name,
                "text_content": df.head(100).to_string(),
            })
        except Exception as e:
            logger.warning(f"Sheet render failed: {e}")
            page_infos.append({
                "page_num":    len(page_infos) + 1,
                "image_path":  None,
                "sheet_name":  sheet_name,
                "text_content": df.head(100).to_string(),
            })
    return page_infos


def process_image(file_path: Path, doc_id: str) -> List[Dict]:
    img = Image.open(str(file_path)).convert("RGB")
    img.thumbnail((1024, 1024))
    page_path = PAGES_DIR / f"{doc_id}_img.png"
    img.save(str(page_path), "PNG")
    return [{
        "page_num": 1, "image_path": str(page_path),
        "width": img.width, "height": img.height, "text_content": "",
    }]


def process_docx(file_path: Path, doc_id: str) -> List[Dict]:
    from docx import Document as DocxDocument
    doc       = DocxDocument(str(file_path))
    full_text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
    page_infos = []
    chunks = [full_text[i:i+2000] for i in range(0, max(len(full_text), 1), 2000)]
    for idx, chunk in enumerate(chunks):
        page_path = PAGES_DIR / f"{doc_id}_page_{idx+1}.png"
        try:
            fig, ax = plt.subplots(figsize=(10, 14)); ax.axis('off')
            ax.text(0.05, 0.95, chunk, transform=ax.transAxes, fontsize=9,
                    verticalalignment='top', fontfamily='monospace', wrap=True)
            plt.title(f"{file_path.stem} — Page {idx+1}")
            plt.savefig(str(page_path), dpi=120, bbox_inches='tight', facecolor='white')
            plt.close(fig)
            page_infos.append({
                "page_num":    idx + 1,
                "image_path":  str(page_path),
                "text_content": chunk,
            })
        except Exception as e:
            page_infos.append({
                "page_num":    idx + 1,
                "image_path":  None,
                "text_content": chunk,
            })
    return page_infos


def ingest_document(file_path: Path, doc_id: str) -> Dict:
    """
    Convert document to page screenshots, embed each with ColPali,
    and store the resulting vector + metadata in ChromaDB.

    ChromaDB entry per page
      id        : "<doc_id>_p<page_num>"  unique string key
      embedding : ColPali mean-pooled float32 vector (provided as list)
      metadata  : doc_id, doc_name, page_num, image_path, text_content
      document  : text_content (ChromaDB full-text field)
    """
    ext  = file_path.suffix.lower()
    meta = load_metadata()

    if ext == '.pdf':
        pages = process_pdf(file_path, doc_id)
    elif ext in ('.xlsx', '.xls', '.csv'):
        pages = process_excel(file_path, doc_id)
    elif ext in ('.png', '.jpg', '.jpeg', '.gif', '.webp', '.bmp', '.tiff'):
        pages = process_image(file_path, doc_id)
    elif ext in ('.doc', '.docx'):
        pages = process_docx(file_path, doc_id)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

    logger.info(f"Embedding {len(pages)} pages and storing in ChromaDB...")
    embedded = 0

    for page in pages:
        img_path = page.get("image_path")
        if not img_path or not os.path.exists(img_path):
            logger.warning(f"Skipping page {page['page_num']} — no image file")
            continue

        # 1. Embed the page screenshot with ColPali
        embedding = embed_image(img_path)
        if embedding is None:
            logger.warning(f"Embedding returned None for page {page['page_num']}")
            continue

        # 2. Build ChromaDB entry
        chroma_id  = f"{doc_id}_p{page['page_num']}"
        # ChromaDB metadata values must be str / int / float / bool.
        # Truncate text_content to 1000 chars to stay within metadata limits.
        text_trunc = page.get("text_content", "")[:1000]

        page_metadata = {
            "doc_id":       doc_id,
            "doc_name":     file_path.name,
            "page_num":     int(page["page_num"]),
            "image_path":   img_path,
            "text_content": text_trunc,
        }

        # 3. Upsert: delete existing entry (no-op if absent), then add fresh
        try:
            CHROMA_COLLECTION.delete(ids=[chroma_id])
        except Exception:
            pass

        CHROMA_COLLECTION.add(
            ids=[chroma_id],
            embeddings=[embedding.tolist()],   # ChromaDB requires a Python list
            metadatas=[page_metadata],
            documents=[text_trunc],
        )

        embedded += 1
        torch.cuda.empty_cache()   # release VRAM after every forward pass

    logger.info(
        f"✅ ChromaDB: {embedded}/{len(pages)} pages stored for '{file_path.name}' "
        f"| Collection total: {CHROMA_COLLECTION.count()} pages"
    )

    doc_meta = {
        "id":         doc_id,
        "name":       file_path.name,
        "type":       ext,
        "size":       file_path.stat().st_size,
        "pages":      pages,
        "page_count": len(pages),
        "status":     "indexed",
        "created_at": datetime.now().isoformat(),
    }
    meta["documents"][doc_id] = doc_meta
    save_metadata(meta)
    return doc_meta


print("✅ Ingestion pipeline ready — ColPali embeddings stored in ChromaDB")


✅ Ingestion pipeline ready — ColPali embeddings stored in ChromaDB


In [5]:
# =======================================================================
# CELL 7 — VRAM Manager  +  Qwen2.5-VL-7B response generation
# =======================================================================

import gc, torch, os, asyncio, base64
from typing import Optional, List, Dict

def free_vram():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free = torch.cuda.mem_get_info()[0] / 1e9
    logger.info(f"VRAM after cleanup: {free:.2f} GB free")
    return free

def offload_colpali():
    global COLPALI_MODEL
    if COLPALI_MODEL is not None:
        try:
            COLPALI_MODEL.to("cpu")
            gc.collect(); torch.cuda.empty_cache()
            logger.info(f"ColPali offloaded to CPU. Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
        except Exception as e:
            logger.warning(f"ColPali offload failed: {e}")

def reload_colpali():
    global COLPALI_MODEL
    if COLPALI_MODEL is not None:
        try:
            COLPALI_MODEL.to("cuda")
            torch.cuda.synchronize()
            logger.info(f"ColPali reloaded to GPU. Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
        except Exception as e:
            logger.warning(f"ColPali reload failed: {e}")


import subprocess
subprocess.run(
    ["pip", "install", "-q",
     "transformers>=4.49.0",
     "qwen-vl-utils",
     "accelerate>=0.27.0"],
    capture_output=True
)

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from PIL import Image

VL_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

if "VL_MODEL" in dir() and VL_MODEL is not None:
    del VL_MODEL
gc.collect(); torch.cuda.empty_cache()
print(f"Free VRAM before Qwen load: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

logger.info("Loading Qwen2.5-VL-7B in full precision (bfloat16)...")
VL_MODEL = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VL_MODEL_ID,
    device_map="cuda",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
VL_MODEL.eval()

VL_PROCESSOR = AutoProcessor.from_pretrained(
    VL_MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28,
)
print(f"✅ Qwen2.5-VL ready! Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")


def vl_generate(messages: list, max_new_tokens: int = 1024) -> str:
    text = VL_PROCESSOR.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = VL_PROCESSOR(
        text=[text],
        images=image_inputs if image_inputs else None,
        videos=video_inputs if video_inputs else None,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        output_ids = VL_MODEL.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
    return VL_PROCESSOR.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]


def image_to_base64(image_path: str) -> Optional[str]:
    try:
        with open(image_path, "rb") as f:
            data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/png;base64,{data}"
    except Exception:
        return None


async def generate_response(
    query: str,
    context_pages: List[Dict],
    history: List[Dict],
    model: str = "qwen2.5-vl",
) -> Dict:
    system_prompt = (
        "You are BuildMarshalAI, an intelligent construction document assistant.\n"
        "You help users understand and analyze uploaded project documents "
        "(PDFs, spreadsheets, images, drawings).\n"
        "- Answer ONLY based on the provided document context.\n"
        "- If the answer is not in the documents, say so clearly.\n"
        "- Cite the document name and page number for each piece of information.\n"
        "- Use markdown headers, bullet points, and tables where appropriate.\n"
        "- Describe drawings or diagrams in detail if visible."
    )

    messages = [{"role": "system", "content": system_prompt}]

    for msg in history[-6:]:
        messages.append({
            "role":    msg.get("role", "user"),
            "content": msg.get("content", ""),
        })

    user_content = []
    images_added = 0
    for page in context_pages:
        if images_added >= 4:
            break
        if page.get("image_path") and os.path.exists(page["image_path"]):
            user_content.append({"type": "image", "image": page["image_path"]})
            images_added += 1

    context_text = "Relevant document pages:\n"
    for i, page in enumerate(context_pages):
        context_text += (
            f"\n[Source {i+1}: {page['doc_name']}, Page {page['page']}"
            f" | similarity={page.get('score', 0):.3f}]"
        )
        if page.get("text_content"):
            context_text += f"\n{page['text_content'][:800]}"

    user_content.append({
        "type": "text",
        "text": f"{context_text}\n\nUser question: {query}",
    })
    messages.append({"role": "user", "content": user_content})

    try:
        loop     = asyncio.get_event_loop()
        response = await loop.run_in_executor(
            None, lambda: vl_generate(messages, max_new_tokens=1024)
        )

        sources = []
        for page in context_pages:
            source = {
                "doc_name":  page["doc_name"],
                "doc_id":    page["doc_id"],
                "page":      page["page"],
                "score":     page.get("score", 0.0),
                "image_url": None,
            }
            if page.get("image_path"):
                source["image_url"] = image_to_base64(page["image_path"])
            sources.append(source)

        return {"response": response, "sources": sources, "model_used": "qwen2.5-vl-7b-bf16"}

    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        raise HTTPException(
            status_code=500,
            detail="GPU out of memory. Try a shorter query or fewer documents.",
        )
    except Exception as e:
        logger.error(f"Inference error: {e}")
        raise HTTPException(status_code=500, detail=f"Inference error: {str(e)}")


print("✅ Query pipeline ready — ColPali (ChromaDB retrieval) + Qwen2.5-VL (generation)")

2026-04-04 21:19:58,532 | INFO | Loading Qwen2.5-VL-7B in full precision (bfloat16)...
2026-04-04 21:19:58,669 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-04 21:19:58,677 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-7B-Instruct/cc594898137f460bfe9f0759e9844b3ce807cfb5/config.json "HTTP/1.1 200 OK"


Free VRAM before Qwen load: 72.55 GB


2026-04-04 21:19:58,788 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-04 21:19:58,795 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-7B-Instruct/cc594898137f460bfe9f0759e9844b3ce807cfb5/config.json "HTTP/1.1 200 OK"
2026-04-04 21:19:58,905 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-04-04 21:19:59,008 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct/resolve/main/model.safetensors.index.json "HTTP/1.1 307 Temporary Redirect"
2026-04-04 21:19:59,015 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-7B-Instruct/cc594898137f460bfe9f0759e9844b3ce807cfb5/model.safetensors.index.json "HTTP/1.1 200 OK"
2026-04-04 21:19:59,120 | INFO | HTTP Request: GET https://huggingface.co/api/models/

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

2026-04-04 21:20:02,368 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-04 21:20:02,377 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-7B-Instruct/cc594898137f460bfe9f0759e9844b3ce807cfb5/generation_config.json "HTTP/1.1 200 OK"
2026-04-04 21:20:02,485 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-04-04 21:20:02,601 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-04 21:20:02,609 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-7B-Instruct/cc594898137f460bfe9f0759e9844b3ce807cfb5/preprocessor_config.json "HTTP/1.1 200 OK"
2026-04-04 21:20:02,729 | INFO | HTTP Request: HEAD https://hugging

✅ Qwen2.5-VL ready! Free VRAM: 55.95 GB
✅ Query pipeline ready — ColPali (ChromaDB retrieval) + Qwen2.5-VL (generation)


In [6]:
# =======================================================================
# CELL 8 — FastAPI application
# =======================================================================

app = FastAPI(
    title="BuildMarshalAI Backend",
    description="ColPali (ChromaDB vector store) + Qwen2.5-VL document chatbot",
    version="2.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/api/health")
async def health_check():
    return {
        "status":          "healthy",
        "gpu_available":   torch.cuda.is_available(),
        "gpu_name":        torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "chroma_pages":    CHROMA_COLLECTION.count(),
        "documents_count": len(load_metadata().get("documents", {})),
    }


@app.get("/api/status")
async def get_status():
    meta  = load_metadata()
    docs  = meta.get("documents", {})
    total = sum(d.get("page_count", 0) for d in docs.values())
    return {
        "documents_count": len(docs),
        "total_pages":     total,
        "chroma_pages":    CHROMA_COLLECTION.count(),
        "retriever":       "colpali-v1.2 + chromadb",
        "generator":       "qwen2.5-vl-7b-4bit",
    }


@app.post("/api/upload")
async def upload_document(file: UploadFile = File(...), doc_id: str = Form(None)):
    if not doc_id:
        doc_id = str(uuid.uuid4())[:12]
    ext     = Path(file.filename).suffix.lower()
    allowed = {
        '.pdf', '.xlsx', '.xls', '.csv',
        '.png', '.jpg', '.jpeg', '.gif', '.webp', '.bmp', '.tiff',
        '.doc', '.docx',
    }
    if ext not in allowed:
        raise HTTPException(400, detail=f"Unsupported file type: {ext}")
    save_path = DOCS_DIR / f"{doc_id}{ext}"
    try:
        save_path.write_bytes(await file.read())
    except Exception as e:
        raise HTTPException(500, detail=f"Could not save file: {e}")
    try:
        doc_meta = ingest_document(save_path, doc_id)
        return {
            "id":      doc_id,
            "name":    file.filename,
            "pages":   doc_meta["page_count"],
            "status":  "indexed",
            "message": (
                f"Indexed {doc_meta['page_count']} pages into ChromaDB "
                f"(collection total: {CHROMA_COLLECTION.count()})"
            ),
        }
    except Exception as e:
        logger.error(f"Ingestion error: {e}")
        return {"id": doc_id, "name": file.filename, "pages": 0,
                "status": "error", "message": str(e)}


@app.get("/api/documents")
async def list_documents():
    meta = load_metadata()
    return {"documents": [
        {
            "id":         did,
            "name":       d["name"],
            "type":       d["type"],
            "size":       d.get("size", 0),
            "pages":      d.get("page_count", 0),
            "status":     d.get("status", "indexed"),
            "created_at": d.get("created_at"),
        }
        for did, d in meta.get("documents", {}).items()
    ]}


@app.get("/api/documents/{doc_id}/status")
async def document_status(doc_id: str):
    doc = load_metadata().get("documents", {}).get(doc_id)
    if not doc:
        raise HTTPException(404, detail="Document not found")
    return {
        "id":     doc_id,
        "status": doc.get("status", "unknown"),
        "pages":  doc.get("page_count", 0),
    }


@app.delete("/api/documents/{doc_id}")
async def delete_document(doc_id: str):
    meta = load_metadata()
    doc  = meta.get("documents", {}).get(doc_id)
    if not doc:
        raise HTTPException(404, detail="Document not found")

    # 1. Remove page screenshots from disk
    for page in doc.get("pages", []):
        img = page.get("image_path")
        if img and os.path.exists(img):
            os.remove(img)
    for f in DOCS_DIR.glob(f"{doc_id}*"):
        f.unlink()

    # 2. Remove all ChromaDB vectors that belong to this document.
    #    We filter by the "doc_id" metadata field so every page entry
    #    for this document is deleted in a single call.
    try:
        CHROMA_COLLECTION.delete(where={"doc_id": doc_id})
        logger.info(
            f"ChromaDB: removed entries for doc_id={doc_id} "
            f"| Collection now has {CHROMA_COLLECTION.count()} pages"
        )
    except Exception as e:
        logger.warning(f"ChromaDB delete failed for {doc_id}: {e}")

    del meta["documents"][doc_id]
    save_metadata(meta)
    return {"message": "Document deleted", "id": doc_id}


@app.post("/api/chat")
async def chat(request: Request):
    body  = await request.json()
    query = body.get("query", "").strip()
    if not query:
        raise HTTPException(400, detail="Query cannot be empty")

    history = body.get("history", [])
    model   = body.get("model", "qwen2.5-vl")
    top_k   = min(body.get("top_k", 5), 20)

    logger.info(f"Query: {query[:80]}...")

    # Step 1: Retrieve top-k pages with ColPali + ChromaDB
    context_pages = retrieve_context(query, top_k=top_k)
    logger.info(f"Retrieved {len(context_pages)} pages from ChromaDB")

    # Step 2: Generate answer with Qwen2.5-VL using retrieved pages
    return await generate_response(query, context_pages, history, model=model)


@app.get("/api/pages/{doc_id}/{page_num}")
async def get_page_image(doc_id: str, page_num: int):
    doc = load_metadata().get("documents", {}).get(doc_id)
    if not doc:
        raise HTTPException(404, detail="Document not found")
    for page in doc.get("pages", []):
        if page.get("page_num") == page_num:
            img_path = page.get("image_path")
            if img_path and os.path.exists(img_path):
                with open(img_path, "rb") as f:
                    return StreamingResponse(BytesIO(f.read()), media_type="image/png")
    raise HTTPException(404, detail="Page not found")


@app.get("/api/chroma/stats")
async def chroma_stats():
    """Inspect ChromaDB collection: total count + sample of stored entries."""
    count  = CHROMA_COLLECTION.count()
    sample = []
    if count > 0:
        peek = CHROMA_COLLECTION.peek(limit=5)
        for i, meta in enumerate(peek.get("metadatas", [])):
            sample.append({
                "id":       peek["ids"][i],
                "doc_name": meta.get("doc_name"),
                "page_num": meta.get("page_num"),
                "doc_id":   meta.get("doc_id"),
            })
    return {"total_pages": count, "sample": sample}


print("✅ FastAPI app defined — ChromaDB-backed endpoints ready")


✅ FastAPI app defined — ChromaDB-backed endpoints ready


In [ ]:
def start_server():
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    port       = 8000
    public_url = ngrok.connect(port)

    print("\n" + "=" * 60)
    print("🏗️  BuildMarshalAI Backend is LIVE!")
    print("=" * 60)
    print(f"\n🌐 PUBLIC URL : {public_url.public_url}")
    print(f"📋 Copy into your frontend Settings panel")
    print(f"🔗 API Docs   : {public_url.public_url}/docs")
    print(f"💚 Health     : {public_url.public_url}/api/health")
    print(f"📊 ChromaDB   : {public_url.public_url}/api/chroma/stats")
    print("\n" + "=" * 60)
    print("⚠️  URL changes each session. Keep this cell running!")
    print("=" * 60 + "\n")

    uvicorn.run(app, host="0.0.0.0", port=port)

start_server()


2026-04-04 21:20:15,335 | INFO | Updating authtoken for default "config_path" of "ngrok_path": /root/.config/ngrok/ngrok
2026-04-04 21:20:15,348 | INFO | Opening tunnel named: http-8000-3ec9f95b-91de-4fd1-b021-6d7d930c32db
2026-04-04 21:20:15,359 | INFO | t=2026-04-04T21:20:15+0000 lvl=info msg="no configuration paths supplied"
2026-04-04 21:20:15,359 | INFO | t=2026-04-04T21:20:15+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
2026-04-04 21:20:15,360 | INFO | t=2026-04-04T21:20:15+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=nil
2026-04-04 21:20:15,369 | INFO | t=2026-04-04T21:20:15+0000 lvl=info msg="FIPS 140 mode" enabled=false
2026-04-04 21:20:15,373 | INFO | t=2026-04-04T21:20:15+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
2026-04-04 21:20:15,719 | INFO | t=2026-04-04T21:20:15+0000 lvl=info msg="client session established" obj=tunnels.session
2026-04-04 21:20:1


🏗️  BuildMarshalAI Backend is LIVE!

🌐 PUBLIC URL : https://irvin-keratosic-earle.ngrok-free.dev
📋 Copy into your frontend Settings panel
🔗 API Docs   : https://irvin-keratosic-earle.ngrok-free.dev/docs
💚 Health     : https://irvin-keratosic-earle.ngrok-free.dev/api/health
📊 ChromaDB   : https://irvin-keratosic-earle.ngrok-free.dev/api/chroma/stats

⚠️  URL changes each session. Keep this cell running!



INFO:     Started server process [1148]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
2026-04-04 21:20:27,048 | INFO | t=2026-04-04T21:20:27+0000 lvl=info msg="join connections" obj=join id=c090cd0d6332 l=127.0.0.1:8000 r=103.94.134.85:59716


INFO:     103.94.134.85:0 - "OPTIONS /api/health HTTP/1.1" 200 OK
INFO:     103.94.134.85:0 - "GET /api/health HTTP/1.1" 200 OK
INFO:     103.94.134.85:0 - "OPTIONS /api/documents/mnkt3c7dk41qvs HTTP/1.1" 200 OK


2026-04-04 21:20:32,205 | INFO | ChromaDB: removed entries for doc_id=mnkt3c7dk41qvs | Collection now has 0 pages


...

INFO:     103.94.134.85:0 - "GET /api/health HTTP/1.1" 200 OK


2026-04-04 21:31:51,942 | INFO | t=2026-04-04T21:31:51+0000 lvl=info msg="join connections" obj=join id=2131060e4da4 l=127.0.0.1:8000 r=103.94.134.85:59786


INFO:     103.94.134.85:0 - "GET /api/health HTTP/1.1" 200 OK


2026-04-04 21:32:31,148 | INFO | t=2026-04-04T21:32:31+0000 lvl=info msg="join connections" obj=join id=519500cf50bb l=127.0.0.1:8000 r=103.94.134.85:59786


INFO:     103.94.134.85:0 - "GET /api/health HTTP/1.1" 200 OK
